## Setup and Imports

In [ ]:
# Standard Library Imports
import sys
import os
import json
import warnings
from pathlib import Path
from datetime import datetime
warnings.filterwarnings('ignore')

# Add src directory to path
project_root = Path().resolve().parents[1]
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
    
# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Great Expectations
import great_expectations as ge


# Project utilities
from utils.helpers import load_orders, set_style, save_figure

set_style()

print("All imports successful!")
print(f'Great Expectations version: {ge.__version__}')
print(f"Project root: {project_root}")

## Load the Clean Master Dataset

In [ ]:
# Load the clean master dataset
processed_path = project_root / 'data' / 'processed'
input_file = processed_path / 'orders_clean_master.csv'

# Load as a standard pandas DataFrame first for inspection
df = pd.read_csv(input_file, parse_dates=['PURCHASE_TS', 'SHIP_TS'])

print('Clean master dataset loaded')
print(f'Rows:    {len(df):,}')
print(f'Columns: {len(df.columns)}')
print(f'\nColumns present:')
for col in df.columns:
    null_count = df[col].isnull().sum()
    print(f'  {col:<35} dtype: {str(df[col].dtype):<15} nulls: {null_count:,}')

In [ ]:
# Create a Great Expectations dataset from the pandas DataFrame

ge_df = ge.from_pandas(df)

print('Great Expectations dataset created ✓')
print(f'Type: {type(ge_df)}')

## Define Expectation Suite

In [ ]:
results = []

def record(category, name, result_obj):
    """Helper function to record expectation results in a consistent format."""
    success = result_obj['success']
    results.append({
        'category':        category,
        'expectation':     name,
        'passed':          success,
        'status':          '✅' if success else '❌',
    })
    status = '✅ PASS' if success else '❌ FAIL'
    print(f'{status} {name}')
    return result_obj

print('Result recorder ready ✓')

In [ ]:
# Category 1: Schema Validation

print('\n Category: Schema Validation')

expected_columns = [
        'USER_ID', 'ORDER_ID', 'PURCHASE_TS', 'SHIP_TS',
    'PRODUCT_NAME', 'PRODUCT_ID', 'USD_PRICE',
    'PURCHASE_PLATFORM', 'MARKETING_CHANNEL',
    'ACCOUNT_CREATION_METHOD', 'COUNTRY_CODE',
    'REGION', 'FULFILMENT_DAYS', 'IS_ANOMALY',
    'IS_ZERO_PRICE', 'IS_PRICE_OUTLIER'
]

for col in expected_columns:
    record(
        'Schema Validation', 
        f'Column "{col}" exists', 
        ge_df.expect_column_to_exist(col))

In [ ]:
# Category 2: Completeness

print('\n Category: Completeness')

must_be_complete = [
    'ORDER_ID', 'USER_ID', 'PRODUCT_NAME',
    'PRODUCT_ID', 'PURCHASE_PLATFORM'
]
for col in must_be_complete:
    record(
        'Completeness', 
        f'Column "{col}" has no nulls', 
        ge_df.expect_column_values_to_not_be_null(col)
    )
    
# USD_PRICE should be complete for paid orders, but can be null for free items so we allow up to 0.1% nulls
record(
    'Completeness', 
    'USD_PRICE is mostly complete (<=0.1% nulls)', 
    ge_df.expect_column_values_to_not_be_null('USD_PRICE', mostly=0.999)
)

# MARKETING_CHANNEL can be null for organic purchases, so we allow up to 1% nulls
record(
    'Completeness', 
    'MARKETING_CHANNEL is mostly complete (<=1% nulls)', 
    ge_df.expect_column_values_to_not_be_null('MARKETING_CHANNEL', mostly=0.99)
)

In [ ]:
# Category 3: Value Ranges

print('\n Category: Value Ranges')

# USD_PRICE: must be >= 0 (no negative prices)
# We allow $0 here because we flag them separately with IS_ZERO_PRICE
record(
    'Value Range',
    'USD_PRICE >= 0 (no negative prices)',
    ge_df.expect_column_values_to_be_between(
        'USD_PRICE', min_value=0, max_value=None
    )
)

# USD_PRICE: max should not exceed $4,000
# Our highest observed price was $3,146 — $4,000 gives a safe buffer
record(
    'Value Range',
    'USD_PRICE <= $4,000 (no implausible prices)',
    ge_df.expect_column_values_to_be_between(
        'USD_PRICE', min_value=None, max_value=4000, mostly=0.999
    )
)

# FULFILMENT_DAYS: should be between -200 and 400
# Our most extreme anomaly was -149 days, maximum normal was 319 days
record(
    'Value Range',
    'FULFILMENT_DAYS between -200 and 400',
    ge_df.expect_column_values_to_be_between(
        'FULFILMENT_DAYS', min_value=-200, max_value=400, mostly=0.999
    )
)

# PURCHASE_TS: all dates should fall within the dataset period 2018-2022
record(
    'Value Range',
    'PURCHASE_TS within dataset period (2018-2022)',
    ge_df.expect_column_values_to_be_between(
        'PURCHASE_TS',
        min_value='2018-01-01',
        max_value='2022-12-31',
        mostly=0.999
    )
)

In [ ]:
# Category 4: Valid Categories

print('\n Category: Valid Categories')

# PURCHASE_PLATFORM: only two valid values
record(
    'Valid Categories',
    'PURCHASE_PLATFORM only contains known values',
    ge_df.expect_column_values_to_be_in_set(
        'PURCHASE_PLATFORM',
        ['website', 'mobile app']
    )
)

# MARKETING_CHANNEL: only known channels allowed
record(
    'Valid Categories',
    'MARKETING_CHANNEL only contains known values',
    ge_df.expect_column_values_to_be_in_set(
        'MARKETING_CHANNEL',
        ['direct', 'email', 'affiliate', 'social media', 'unknown', None]
    )
)

# PRODUCT_NAME: after entity resolution, only 8 canonical names should exist
canonical_products = [
    '27in 4K gaming monitor',
    'Nintendo Switch',
    'Sony PlayStation 5 Bundle',
    'JBL Quantum 100 Gaming Headset',
    'Dell Gaming Mouse',
    'Acer Nitro V Gaming Laptop',
    'Lenovo IdeaPad Gaming 3',
    'Razer Pro Gaming Headset',
]
record(
    'Valid Categories',
    'PRODUCT_NAME: no legacy names remain after entity resolution',
    ge_df.expect_column_values_to_be_in_set(
        'PRODUCT_NAME',
        canonical_products
    )
)

# REGION: only valid region codes
record(
    'Valid Categories',
    'REGION only contains known values',
    ge_df.expect_column_values_to_be_in_set(
        'REGION',
        ['Americas', 'EMEA', 'APAC', 'LATAM', 'Other', None]
    )
)

# ACCOUNT_CREATION_METHOD: only known methods
record(
    'Valid Categories',
    'ACCOUNT_CREATION_METHOD only contains known values',
    ge_df.expect_column_values_to_be_in_set(
        'ACCOUNT_CREATION_METHOD',
        ['desktop', 'mobile', 'tablet', 'tv', 'unknown', None]
    )
)

In [ ]:
# Category 5: Uniqueness and Business Rules

print('\n Category: Business Rules')

# ORDER_ID must be unique — each order is a distinct transaction
# Duplicate order IDs would mean we are double-counting revenue
record(
    'Business Rules',
    'ORDER_ID values are unique (no duplicate orders)',
    ge_df.expect_column_values_to_be_unique('ORDER_ID')
)

# Row count should be within expected range
# We expect ~21,864 rows — allow +/- 100 for any minor processing differences
record(
    'Business Rules',
    'Row count within expected range (21,700 - 22,000)',
    ge_df.expect_table_row_count_to_be_between(21700, 22000)
)

# Column count should be exactly what we expect
record(
    'Business Rules',
    f'Column count is at least 16',
    ge_df.expect_table_column_count_to_be_between(16, 25)
)

# IS_ANOMALY must be boolean (True/False only)
record(
    'Business Rules',
    'IS_ANOMALY only contains True/False values',
    ge_df.expect_column_values_to_be_in_set('IS_ANOMALY', [True, False])
)

# IS_ZERO_PRICE must be boolean
record(
    'Business Rules',
    'IS_ZERO_PRICE only contains True/False values',
    ge_df.expect_column_values_to_be_in_set('IS_ZERO_PRICE', [True, False])
)

In [ ]:
# Category 6: Statistical properties 

print('\n Category 6: Statistical Properties')

# Median USD_PRICE should be close to $168 (the Nintendo Switch price)
# We allow a $50 band either side
record(
    'Statistical',
    'Median USD_PRICE between $118 and $218',
    ge_df.expect_column_median_to_be_between('USD_PRICE', 118, 218)
)

# Mean USD_PRICE should be between $200 and $350
# We found $281 — allowing a reasonable band
record(
    'Statistical',
    'Mean USD_PRICE between $200 and $350',
    ge_df.expect_column_mean_to_be_between('USD_PRICE', 200, 350)
)

# Anomaly rate should be between 8% and 11%
# We found 9.1% — a significant deviation would indicate a processing error
anomaly_rate = df['IS_ANOMALY'].mean()
record(
    'Statistical',
    'IS_ANOMALY rate between 8% and 11%',
    ge_df.expect_column_mean_to_be_between('IS_ANOMALY', 0.08, 0.11)
)

# Number of unique products should be exactly 9 after entity resolution
record(
    'Statistical',
    'Exactly 9 unique product names after entity resolution',
    ge_df.expect_column_unique_value_count_to_be_between('PRODUCT_NAME', 8, 9)
)

## Validation Results Summary

In [ ]:
# Build a clean results DataFrame
results_df = pd.DataFrame(results)

total      = len(results_df)
passed     = results_df['passed'].sum()
failed     = total - passed
pass_rate  = passed / total * 100

print('  VALIDATION RESULTS SUMMARY')
print(f'  Total expectations: {total}')
print(f'  Passed:             {passed}  ({pass_rate:.1f}%)')
print(f'  Failed:             {failed}')
print()

# Results by category
print('Results by category:')
category_summary = results_df.groupby('category').agg(
    total  = ('passed', 'count'),
    passed = ('passed', 'sum')
).assign(failed = lambda x: x['total'] - x['passed'])
print(category_summary.to_string())
print()

# Show any failures in detail
failures = results_df[~results_df['passed']]
if len(failures) > 0:
    print(f'\n❌ FAILED EXPECTATIONS ({len(failures)}):')
    for _, row in failures.iterrows():
        print(f'  [{row["category"]}] {row["expectation"]}')
else:
    print('✅ All expectations passed — dataset is clean and ready for analysis.')

## Visualise Validation Results

In [ ]:
# Chart 1: Pass/fail overview by category
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: stacked bar chart by category
cat_data = results_df.groupby('category')['passed'].agg(['sum', 'count'])
cat_data['failed'] = cat_data['count'] - cat_data['sum']
cat_data = cat_data.sort_values('count', ascending=True)

axes[0].barh(cat_data.index, cat_data['sum'],
             color='#2E75B6', alpha=0.85, label='Passed')
axes[0].barh(cat_data.index, cat_data['failed'],
             left=cat_data['sum'], color='#C00000', alpha=0.85, label='Failed')

for i, (idx, row) in enumerate(cat_data.iterrows()):
    axes[0].text(row['count'] + 0.1, i,
                 f"{row['sum']}/{row['count']}",
                 va='center', fontsize=9)

axes[0].set_title('Validation results by category', fontsize=12, fontweight='medium')
axes[0].set_xlabel('Number of expectations')
axes[0].legend(fontsize=9)

# Right: overall pass/fail donut chart
sizes  = [passed, failed]
colors = ['#2E75B6', '#C00000']
labels = [f'Passed ({passed})', f'Failed ({failed})']

wedges, texts, autotexts = axes[1].pie(
    sizes, colors=colors, labels=labels,
    autopct='%1.0f%%', startangle=90,
    wedgeprops={'width': 0.5}  # donut style
)
axes[1].set_title(f'Overall: {pass_rate:.0f}% pass rate', fontsize=12, fontweight='medium')

fig.suptitle('Great Expectations validation results — GameZone clean master dataset',
             fontsize=13, fontweight='medium', y=1.02)

save_figure(fig, '04_validation_results_overview.png')
plt.show()

In [ ]:
# Chart 2: Individual expectation results
# Shows every single expectation as a pass/fail dot
# Useful for quickly identifying which specific checks failed

fig, ax = plt.subplots(figsize=(10, max(8, len(results_df) * 0.3)))

colours = ['#2E75B6' if p else '#C00000' for p in results_df['passed']]
y_positions = range(len(results_df) - 1, -1, -1)  # reverse so first is at top

ax.scatter(results_df['passed'].astype(int), y_positions,
           c=colours, s=80, zorder=3)

for i, (_, row) in enumerate(results_df.iterrows()):
    y = len(results_df) - 1 - i
    ax.text(1.05, y, row['expectation'], va='center', fontsize=7.5)
    ax.text(-0.05, y, row['category'], va='center',
            fontsize=7, color='gray', ha='right')

ax.set_xlim(-0.5, 1.5)
ax.set_xticks([0, 1])
ax.set_xticklabels(['FAIL', 'PASS'])
ax.set_yticks([])
ax.axvline(0.5, color='#CCCCCC', linewidth=0.8, linestyle='--')
ax.set_title('All expectations — individual pass/fail results',
             fontsize=12, fontweight='medium')
ax.grid(axis='y', alpha=0.3)

save_figure(fig, '04_validation_individual_results.png')
plt.show()

## Save Validation Report

In [ ]:
# Save the results DataFrame as a CSV
report_path = project_root / 'data' / 'processed'
os.makedirs(report_path, exist_ok=True)

# Add metadata columns
results_df['run_date']    = datetime.now().strftime('%Y-%m-%d')
results_df['dataset']     = 'orders_clean_master.csv'
results_df['total_rows']  = len(df)

output_path = report_path / 'validation_results.csv'
results_df.to_csv(output_path, index=False)

print(f'Validation report saved → data/processed/validation_results.csv')
print(f'  Expectations run: {total}')
print(f'  Passed:           {passed}')
print(f'  Failed:           {failed}')
print(f'  Run date:         {datetime.now().strftime("%Y-%m-%d")}')

## Findings and Summary

In [ ]:
print('  NOTEBOOK 4 FINDINGS — GREAT EXPECTATIONS VALIDATION')
print()
print('PURPOSE')
print('  This notebook runs a formal validation suite on the clean')
print('  master dataset produced by Notebooks 1-3. It confirms that')
print('  all quality fixes were applied correctly and the dataset')
print('  meets the minimum standards required for downstream analysis.')
print()
print(f'RESULT: {passed}/{total} expectations passed ({pass_rate:.0f}% pass rate)')
print()
print('EXPECTATIONS VALIDATED')
print('  Schema:          All expected columns present')
print('  Completeness:    Critical columns have no unexpected nulls')
print('  Value ranges:    Prices, dates, fulfilment days within bounds')
print('  Valid categories: Only known platforms, channels, products')
print('  Business rules:  ORDER_ID unique, row count as expected')
print('  Statistical:     Key metrics match investigation findings')
print()

if failed > 0:
    print('FAILED EXPECTATIONS — action required:')
    for _, row in failures.iterrows():
        print(f'  ❌ [{row["category"]}] {row["expectation"]}')
    print()
    print('  Review each failure carefully. Either the data has a')
    print('  genuine issue that needs fixing, or the expectation')
    print('  threshold needs adjusting based on new knowledge.')
else:
    print('✅ All expectations passed.')
    print('  The clean master dataset is validated and ready for:')
    print('  - Project 02: Customer Lifetime Value Segmentation')
    print('  - Project 03: Demand Forecasting')
    print('  - Project 04: Churn & Survival Analysis')

print()
print('WHAT THIS MEANS FOR THE PORTFOLIO')
print('  Having a formal validation suite demonstrates that this')
print('  project follows production data engineering standards —')
print('  not just exploratory analysis. The expectation suite can')
print('  be re-run any time the dataset is refreshed to immediately')
print('  catch any new quality issues.')
print()
print('Notebook 4 complete ✓')
print('Project 01 — Data Quality & Anomaly Audit COMPLETE')
print()
print('Four notebooks completed:')
print('  ✅ Notebook 1: Temporal Anomaly Investigation')
print('  ✅ Notebook 2: Price Forensics')
print('  ✅ Notebook 3: Entity Resolution')
print('  ✅ Notebook 4: Great Expectations Validation')
print()
print('Next: Build the Excel data quality report, then')
print('      merge feature/project-01-audit → dev via PR')

In [ ]:
print('All unique REGION values:')
print(df['REGION'].value_counts(dropna=False))

In [ ]:
dupes = df[df['ORDER_ID'].duplicated(keep=False)].sort_values('ORDER_ID')
print(f'Duplicate ORDER_IDs: {df["ORDER_ID"].duplicated().sum()}')
print(dupes[['ORDER_ID', 'USER_ID', 'PRODUCT_NAME', 'USD_PRICE', 'PURCHASE_TS']].head(20))

In [ ]:
# Which countries have no region assigned?
no_region = df[df['REGION'].isna()]
print(f'Orders with no region: {len(no_region):,}')
print()
print('Top 20 country codes with no region:')
print(no_region['COUNTRY_CODE'].value_counts().head(20))

In [ ]:
# Are ALL duplicates from January 2020?
dupes = df[df['ORDER_ID'].duplicated(keep=False)]
print('Duplicate purchase dates:')
print(dupes['PURCHASE_TS'].dt.to_period('M').value_counts().sort_index())
print()
print('Duplicate products:')
print(dupes['PRODUCT_NAME'].value_counts())
print()
print(f'Total duplicate records: {len(dupes):,}')
print(f'Unique duplicate order IDs: {dupes["ORDER_ID"].nunique()}')

In [ ]:
print('ADDITIONAL FINDINGS FROM VALIDATION FAILURES')
print()

# Finding A: REGION column gaps
no_region = df[df['REGION'].isna()]
north_america = df[df['REGION'] == 'North America']
corrupt = df[df['REGION'] == 'X.x']

print('FINDING A — REGION column has three issues:')
print()
print(f'  1. {len(no_region):,} orders have no region assigned (NaN)')
print(f'     Root cause: US ({no_region["COUNTRY_CODE"].value_counts().iloc[0]:,} orders)')
print(f'     was absent from the region lookup sheet entirely.')
print(f'     Impact: any regional analysis would silently exclude')
print(f'     the largest market in the dataset.')
print()
print(f'  2. {len(north_america):,} orders use "North America" instead of "Americas"')
print(f'     Root cause: inconsistency in the region lookup sheet.')
print(f'     Fix: standardise to "Americas".')
print()
print(f'  3. {len(corrupt):,} orders have corrupt region value "X.x"')
print(f'     Root cause: unknown — likely a lookup table entry error.')
print(f'     Fix: reclassify manually or set to "Other".')
print()

# Finding B: Duplicate ORDER_IDs
dupes = df[df['ORDER_ID'].duplicated(keep=False)]
print(f'FINDING B — 145 duplicate ORDER_IDs, all in January 2020:')
print()
print(f'  Products affected:')
for product, count in dupes['PRODUCT_NAME'].value_counts().items():
    print(f'    {product}: {count} records ({count//2} duplicate pairs)')
print()
print(f'  All duplicates are exact copies — same user, same price,')
print(f'  same date, same order ID. This is consistent with a batch')
print(f'  reprocessing event in January 2020 that wrote every')
print(f'  affected order twice.')
print()
print(f'  Fix: deduplicate by keeping the first occurrence of each ORDER_ID.')

# Apply the fixes
print()
print('APPLYING FIXES')
print()

df_fixed = df.copy()

# Fix 1: Standardise region values
df_fixed['REGION'] = df_fixed['REGION'].replace({
    'North America': 'Americas',
    'X.x': 'Other'
})

# Fix 2: Fill US and other missing regions
region_fixes = {
    'US': 'Americas', 'IE': 'EMEA', 'PR': 'Americas',
    'LB': 'EMEA', 'JM': 'Americas', 'GL': 'Americas',
    'BS': 'Americas', 'TT': 'Americas', 'VI': 'Americas',
    'KY': 'Americas', 'EU': 'EMEA', 'VC': 'Americas',
    'AP': 'APAC', 'KN': 'Americas', 'AI': 'Americas',
    'LC': 'Americas'
}
for country, region in region_fixes.items():
    mask = (df_fixed['COUNTRY_CODE'] == country) & (df_fixed['REGION'].isna())
    df_fixed.loc[mask, 'REGION'] = region

# Fix 3: Remove duplicate ORDER_IDs — keep first occurrence
rows_before = len(df_fixed)
df_fixed = df_fixed.drop_duplicates(subset='ORDER_ID', keep='first')
rows_after = len(df_fixed)

print(f'Region fixes applied:')
print(f'  NaN regions remaining: {df_fixed["REGION"].isna().sum()}')
print(f'  Unique region values:  {sorted(df_fixed["REGION"].dropna().unique().tolist())}')
print()
print(f'Duplicate removal:')
print(f'  Rows before: {rows_before:,}')
print(f'  Rows after:  {rows_after:,}')
print(f'  Removed:     {rows_before - rows_after:,} duplicate records')
print()

# Save the fully corrected master dataset
output_path = project_root / 'data' / 'processed' / 'orders_clean_master.csv'
df_fixed.to_csv(output_path, index=False)
print(f'Updated master dataset saved → orders_clean_master.csv')
print(f'Final row count: {len(df_fixed):,}')

In [ ]:
print('  NOTEBOOK 4 FINDINGS — GREAT EXPECTATIONS VALIDATION')
print()
print('RESULT: 39/41 expectations passed (95% pass rate)')
print('Two failures identified, investigated, and resolved.')
print()
print('FINDING 1 — Schema: all 16 expected columns present ✅')
print('  The clean master dataset contains every column required')
print('  by downstream projects. No schema drift detected.')
print()
print('FINDING 2 — Completeness: all critical columns complete ✅')
print('  ORDER_ID, USER_ID, PRODUCT_NAME, PRODUCT_ID, and')
print('  PURCHASE_PLATFORM have zero null values. USD_PRICE and')
print('  MARKETING_CHANNEL null rates are within expected thresholds.')
print()
print('FINDING 3 — Value ranges: all within expected bounds ✅')
print('  No negative prices. No dates outside 2018-2022.')
print('  Fulfilment days within -200 to 400 day range.')
print()
print('FINDING 4 — Valid categories: confirmed ✅')
print('  Entity resolution from Notebook 3 confirmed — no legacy')
print('  product names remain. Only canonical product names present.')
print()
print('FINDING 5 — Statistical properties confirmed ✅')
print('  Median price: $168. Mean price: $281. Anomaly rate: 9.1%.')
print('  Exactly 9 unique products after entity resolution.')
print('  All statistical expectations passed.')
print()
print('FINDING 6 — REGION column: gaps and inconsistencies ❌ → FIXED')
print('  Validation failure revealed three issues:')
print('  - 10,531 orders had no region (NaN) — root cause: US was')
print('    entirely absent from the region lookup table (10,294 orders)')
print('  - 963 orders used "North America" instead of "Americas"')
print('  - 4 orders had corrupt value "X.x"')
print('  Impact: any regional revenue analysis would have silently')
print('  excluded the US — the largest market in the dataset.')
print('  Resolution: all 15 missing country codes manually mapped,')
print('  "North America" standardised to "Americas", "X.x" → "Other".')
print('  NaN regions remaining after fix: 0')
print()
print('FINDING 7 — Duplicate ORDER_IDs: batch processing event ❌ → FIXED')
print('  145 duplicate ORDER_IDs found, all in January 2020.')
print('  Products affected: Nintendo Switch (98 pairs), 27in 4K')
print('  gaming monitor (37 pairs), PS5 Bundle (10 pairs).')
print('  All duplicates are exact copies — same user, same price,')
print('  same date. Consistent with a batch reprocessing event that')
print('  wrote every January 2020 order twice.')
print('  Resolution: deduplicated by keeping first occurrence.')
print('  Rows removed: 145. Final dataset: 21,719 orders.')
print()
print('RECOMMENDATION')
print('  The two validation failures demonstrate exactly why a formal')
print('  validation suite is valuable — both issues were invisible to')
print('  manual inspection and would have silently corrupted downstream')
print('  analysis. The updated orders_clean_master.csv (21,719 rows)')
print('  with all fixes applied is now ready for Projects 02, 03, 04.')
print()
print('Notebook 4 complete ✓')
print('Project 01 — Data Quality & Anomaly Audit COMPLETE')
print()
print('Four notebooks completed:')
print('  ✅ Notebook 1: Temporal Anomaly Investigation')
print('  ✅ Notebook 2: Price Forensics')
print('  ✅ Notebook 3: Entity Resolution')
print('  ✅ Notebook 4: Great Expectations Validation')
print()
print('Final dataset: orders_clean_master.csv (21,719 rows)')
print('Next: Excel data quality report → PR → Project 02')